This notebook fine tunes PI0.5 VLA model on my LeRobot robotics [dataset](https://huggingface.co/datasets/ankithreddy/so101_pickplace_tools) from hugging face


## 1. Config

In [34]:
import logging
import os
from dotenv import load_dotenv
import numpy as np


load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s")
log = logging.getLogger(__name__)


HF_TOKEN = os.environ["HF_TOKEN"]
WANDB_TOKEN = os.environ["WANDB_TOKEN"]

# LeRobot v3 dataset with parquet + mp4 in HF 
DATASET_ID = "ankithreddy/so101_pickplace_tools"
DATASET_PATH = f"hf://datasets/{DATASET_ID}"
REVISION= "v3" # v3 branch has data converted to v3 format from 2.1

CHECKPOINT_REPO = "ankithreddy/pi05-so101-pickplace"
RUN_NAME = "pi05-so101-pickplace-finetune"


So training the VLA models has two resource intensive tasks 
1. DATA : decode the mp4 data which uses CPU, normalising, transforming  , we ll use ray data to distributed between available cpus
2. TRAINING: Forward/backward passes on a large transformer uses GPU , with Ray Train launches distributed Pytorch workers across GPU's manages DDP synchronization , checkpoining , and fault recovery i.e if a worker dise training resumes from the last checkpoint you dont re run from scratch

## 2. Understanding the data

In [51]:
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

ds_meta = LeRobotDatasetMetadata(DATASET_ID, revision=REVISION)

log.info("robot type    : %s", ds_meta.robot_type)
log.info("total episodes: %d", ds_meta.total_episodes)
log.info("total frames  : %d", ds_meta.total_frames)
log.info("fps           : %d", ds_meta.fps)
log.info("cameras       : %s", ds_meta.camera_keys)
log.info("action dim    : %s", ds_meta.features["action"]["shape"])
log.info("state dim     : %s", ds_meta.features["observation.state"]["shape"])
log.info("tasks         : %s", ds_meta.tasks)
log.info("data path     : %s", ds_meta.info["data_path"])
log.info("video path    : %s", ds_meta.info["video_path"])

for cam in ds_meta.camera_keys:
    log.info("camera %-45s shape: %s", cam, ds_meta.features[cam]["shape"])

# normalization stats — used later in training loop
stats = {
    k: {"mean": ds_meta.stats[k]["mean"], "std": ds_meta.stats[k]["std"]}
    for k in ("action", "observation.state")
}
log.info("stats keys    : %s", list(stats.keys()))

2026-04-05 00:11:07,883  INFO      robot type    : so101_follower
2026-04-05 00:11:07,884  INFO      total episodes: 90
2026-04-05 00:11:07,884  INFO      total frames  : 57019
2026-04-05 00:11:07,885  INFO      fps           : 30
2026-04-05 00:11:07,885  INFO      cameras       : ['observation.images.images.wrist', 'observation.images.images.top']
2026-04-05 00:11:07,885  INFO      action dim    : (6,)
2026-04-05 00:11:07,886  INFO      state dim     : (6,)
2026-04-05 00:11:07,886  INFO      tasks         :                                        task_index
task                                             
pick up screwdriver and put it in box           0
2026-04-05 00:11:07,912  INFO      data path     : data/chunk-{chunk_index:03d}/file-{file_index:03d}.parquet
2026-04-05 00:11:07,912  INFO      video path    : videos/{video_key}/chunk-{chunk_index:03d}/file-{file_index:03d}.mp4
2026-04-05 00:11:07,912  INFO      camera observation.images.images.wrist               shape: (720, 1280,

## 4. Transform data 

  PI0.5 (like most vision models) expects (batch, channels, height, width).
    The raw dataset stores images as (height, width, channels) uint8.
   

In [46]:
def transpose_images(batch:dict, camera_keys: list[str]) -> dict:
    """Convert camera images from HWC uint8 to CHW float32.
       LeRobot stores images as (H, W, C) uint8.
       PI0.5 expects (C, H, W) float32.
    """
    result = dict(batch)
    for key in camera_keys:
        result[key] = np.transpose(np.stack(batch[key]),(0,3,1,2)).astype(np.float32)
    return result
    

## 5. Connect to Ray

In [6]:
import ray

ray.init(
    runtime_env={
        "py_executable":"uv run",
        "working_dir":".",
        "env_vars": {"HF_TOKEN": HF_TOKEN,"WANDB_API_KEY": WANDB_TOKEN},
        
    },
    ignore_reinit_error=True,
)

2026-04-03 22:52:31,533	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-04-03 22:52:37,459	INFO worker.py:2013 -- Started a local Ray instance.
2026-04-03 22:52:37,472	INFO packaging.py:691 -- Creating a file package for local module '.'.
2026-04-03 22:52:37,475	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_be01ee34e1c8ff8b.zip' (0.00MiB) to Ray cluster...
2026-04-03 22:52:37,476	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_be01ee34e1c8ff8b.zip'.
/Users/ankithreddy/Desktop/mlopsaiprojectsnres/vla-distributed-training/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.war

Python version:,3.12.0
Ray version:,2.54.1


In [7]:
print(ray.cluster_resources())


{'object_store_memory': 2147483648.0, 'node:__internal_head__': 1.0, 'CPU': 8.0, 'node:127.0.0.1': 1.0, 'memory': 8844853248.0}


## 5.  Ray Data : building the preprocessing pipeline
   these functions run on ray data's CPU worker pool, Ray Data calls them automatically as it streams data from hf dataset, keeping gpu workers fed without blocking them on I/O or image decoding

In [56]:
from lerobot_datasource import LeRobotDatasource

source = LeRobotDatasource(f"hf://datasets/{DATASET_ID}@{REVISION}")
image_keys = source.meta.video_keys

ds = (
    ray.data 
    .read_datasource(source) # stream from hf dataset
    .map_batches(transpose_images, batch_size=32, fn_args=(image_keys,)) # HWC int8 -> CHW float
)

log.info("pipeline ready — %d frames, cameras: %s", source.meta.total_frames, image_keys)


2026-04-05 00:24:24,608  INFO      HTTP Request: GET https://huggingface.co/datasets/ankithreddy/so101_pickplace_tools/resolve/v3/meta/info.json "HTTP/1.1 307 Temporary Redirect"
2026-04-05 00:24:24,638  INFO      HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/ankithreddy/so101_pickplace_tools/ed0e998fdc1044aa9209ffef6eb7017b70d2df22/meta%2Finfo.json?%2Fdatasets%2Fankithreddy%2Fso101_pickplace_tools%2Fresolve%2Fv3%2Fmeta%2Finfo.json=&etag=%227a5c8f0610a484579e4561b063e5794162601ade%22 "HTTP/1.1 200 OK"
2026-04-05 00:24:24,689  INFO      HTTP Request: GET https://huggingface.co/datasets/ankithreddy/so101_pickplace_tools/resolve/v3/meta/stats.json "HTTP/1.1 307 Temporary Redirect"
2026-04-05 00:24:24,722  INFO      HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/ankithreddy/so101_pickplace_tools/ed0e998fdc1044aa9209ffef6eb7017b70d2df22/meta%2Fstats.json?%2Fdatasets%2Fankithreddy%2Fso101_pickplace_tools%2Fresolve%2Fv3%2Fmeta%2Fstats.json=&etag=%22f

## 6. Ray Train - distributed training 

In [ ]:
import os
import torch
import util

# Ray Train calls this function once per GPU worker.
# Each worker gets its own copy of the model, optimizer, and data shard.
# Ray handles DDP synchronization, checkpointing, and fault recovery automatically.
def train_loop_per_worker(config: dict):
    import ray.train
    import ray.train.torch
    from ray.air.integrations.wandb import setup_wandb
    from lerobot.policies.factory import make_pre_post_processors

    device = torch.device("cuda")

    # setup wandb — rank 0 logs to wandb, other workers get a disabled run
    wandb = setup_wandb(config, project=RUN_NAME)

    # 1. load PI0.5, apply attention mask patch, freeze backbone
    #    only action/time projection heads are trainable — saves memory and compute
    policy = util.load_pi05_policy()

    # wrap in DDP — Ray handles torch.distributed.init_process_group automatically
    policy = ray.train.torch.prepare_model(policy)

    # 2. optimizer — only updates unfrozen action/time projection heads
    optimizer = torch.optim.AdamW(
        [p for p in policy.parameters() if p.requires_grad],
        lr=config.get("lr", 1e-4),
    )
    # mixed precision scaler — fp16 forward pass, fp32 gradient accumulation
    scaler = torch.amp.GradScaler("cuda")

    # 3. resume from checkpoint if training crashed
    #    ray.train.get_checkpoint() returns None on fresh run, last checkpoint on recovery
    checkpoint = ray.train.get_checkpoint()
    if checkpoint:
        start_epoch, step = util.load_checkpoint(checkpoint, policy, optimizer, scaler)
    else:
        start_epoch, step = 0, 0

    # 4. override normalization — our dataset has mean/std but not quantile stats (q01/q99)
    #    PI0.5 defaults to QUANTILES so we must override to MEAN_STD
    policy.module.config.normalization_mapping = {
        "ACTION": "MEAN_STD",
        "STATE": "MEAN_STD",
        "VISUAL": "IDENTITY",  # images are not normalized
    }

    # preprocessor normalizes action/state and tokenizes the language instruction
    preprocessor, _ = make_pre_post_processors(
        policy.module.config,
        pretrained_path="lerobot/pi05_base",
        dataset_stats=config["stats"],
    )

    batch_size  = int(config.get("batch_size", 1))
    grad_accum  = int(config.get("grad_accum", 1))
    num_epochs  = int(config.get("num_epochs", 1))
    max_len     = int(config.get("max_len", 512))
    num_workers = ray.train.get_context().get_world_size()

    # linear warmup + cosine decay lr schedule
    scheduler = util.build_lr_scheduler(optimizer, config, num_workers, last_step=step)

    # 5. get this worker's shard — Ray splits data across workers automatically
    #    no DistributedSampler needed, Ray handles it
    shard = ray.train.get_dataset_shard("train")

    for epoch in range(start_epoch, num_epochs):
        optimizer.zero_grad(set_to_none=True)
        epoch_loss_sum, epoch_loss_count = 0.0, 0
        accum_count = 0

        # iter_torch_batches streams decoded frames from CPU workers directly to GPU
        for batch in shard.iter_torch_batches(
            batch_size=batch_size,
            collate_fn=util.NumpyToTorchCollate(device),
        ):
            loss_val = util.train_step(policy, batch, preprocessor, max_len, grad_accum, scaler)
            step += 1
            accum_count += 1
            epoch_loss_sum += loss_val
            epoch_loss_count += 1

            if accum_count % grad_accum == 0:
                util.optimizer_step(policy, optimizer, scaler, scheduler)
                accum_count = 0

            if step % 10 == 0:
                log.info("epoch=%d step=%d loss=%.4f lr=%.2e",
                         epoch, step, loss_val, scheduler.get_last_lr()[0])
                wandb.log({"loss": loss_val, "lr": scheduler.get_last_lr()[0], "step": step})

        # flush remaining gradients at epoch end
        if accum_count > 0:
            util.optimizer_step(policy, optimizer, scaler, scheduler)

        avg_loss = epoch_loss_sum / max(epoch_loss_count, 1)
        metrics = {"epoch": epoch, "steps": step, "loss": avg_loss, "lr": scheduler.get_last_lr()[0]}
        wandb.log({"epoch_loss": avg_loss, "epoch": epoch})

        # ray.train.report() is a sync barrier — all workers must call it
        # only rank 0 saves the checkpoint
        if ray.train.get_context().get_world_rank() == 0:
            checkpoint = util.make_checkpoint(policy, optimizer, scaler, epoch, step)
            ray.train.report(metrics, checkpoint=checkpoint)
        else:
            ray.train.report(metrics)

## 7. launch training 

In [ ]:
import ray.train
import ray.train.torch
from ray.air.integrations.wandb import WandbLoggerCallback

# everything passed into train_loop_per_worker on every GPU worker
train_loop_config = {
    "stats":       stats,                    # mean/std normalization stats for action/state
    "total_rows":  source.meta.total_frames, # used by lr scheduler to compute total steps
    "num_epochs":  1,                        # increase for full training run
    "batch_size":  4,                        # per GPU — works for 48GB RTX A6000
    "grad_accum":  2,                        # effective batch = batch_size * grad_accum * num_workers
    "lr":          1e-4,
    "warmup_frac": 0.1,                      # 10% of steps for lr warmup
    "max_len":     512,                      # truncate token sequences to cap GPU memory
}

# 2 GPU workers — each gets 1 RTX A6000, Ray handles DDP sync between them
scaling_config = ray.train.ScalingConfig(num_workers=2, use_gpu=True)

# fault tolerance + wandb logging
# max_failures=1 — if a worker crashes, Ray restarts from last checkpoint automatically
# WandbLoggerCallback — logs everything from ray.train.report() to wandb dashboard
run_config = ray.train.RunConfig(
    name=RUN_NAME,
    storage_path="/tmp/ray_runs",
    failure_config=ray.train.FailureConfig(max_failures=1),
    callbacks=[WandbLoggerCallback(project=RUN_NAME)],
)

# TorchTrainer ties it all together:
# — sends ds (Ray Data pipeline) to workers as shards
# — launches train_loop_per_worker on each GPU
# — handles DDP, checkpointing, fault recovery
trainer = ray.train.torch.TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config=train_loop_config,
    scaling_config=scaling_config,
    run_config=run_config,
    datasets={"train": ds},
)

result = trainer.fit()
log.info("training complete: %s", result)

In [ ]:
if result.checkpoint:
    from huggingface_hub import HfApi
    
    with result.checkpoint.as_directory() as ckpt_dir:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(CHECKPOINT_REPO, repo_type="model", exist_ok=True)
        api.upload_folder(
            folder_path=ckpt_dir,
            repo_id=CHECKPOINT_REPO,
            repo_type="model",
        )
        log.info("checkpoint pushed to: %s", CHECKPOINT_REPO)